# Collaboration and Competition — Unity Tennis

A guided tour of the environment and the trained agents. Training itself runs from the command line rather than in this notebook — a run that solves takes 20+ minutes and gets slower as it improves, which a notebook is a poor place to keep alive.

**Before running this:** install the package and unzip the Unity build into the repository root — see [README.md](README.md).

## 1. Start the environment

In [ ]:
import numpy as np

from tennis.env import TennisEnv

# no_graphics=True trains much faster; set it False to watch the rally.
env = TennisEnv(no_graphics=True, seed=0)

print(f"agents      : {env.num_agents}")
print(f"observation : {env.state_size} per agent")
print(f"action      : {env.action_size} per agent ({env.action_type})")

## 2. The state and action spaces

Each racket sees 24 values — 3 stacked frames of 8 variables describing the position and velocity of the ball and its own racket. Each action is 2 values in `[-1, 1]`: movement toward or away from the net, and jumping.

Note that each agent gets its **own** observation and its **own** reward. They are not sharing a view of the world.

In [ ]:
states = env.reset(train_mode=True)
print(f"states shape: {states.shape}  (one row per agent)")
print(f"\nagent 0's observation:\n{np.round(states[0], 3)}")

## 3. A random policy — the floor to beat

Note how the episode ends: not after a fixed number of steps, but as soon as the ball is not returned. A random policy manages about 17 steps.

**The scoring rule matters here.** An episode's score is the *maximum* of the two agents' totals, not the mean. Using the mean would roughly halve every score and a working agent would never register as solved.

In [ ]:
states = env.reset(train_mode=True)
totals = np.zeros(env.num_agents)
steps = 0

while True:
    actions = np.random.uniform(-1, 1, (env.num_agents, env.action_size))
    states, rewards, dones, _ = env.step(actions)
    totals += rewards
    steps += 1
    if dones.any():
        break

print(f"per-agent totals : {np.round(totals, 3)}")
print(f"episode score    : {totals.max():.3f}   (max, not mean)")
print(f"rally length     : {steps} steps")

Averaged over 50 episodes a random policy scores **0.0154**, with only 7 of those episodes scoring above zero at all. The target is **+0.5** averaged over 100 consecutive episodes.

```bash
python scripts/random_baseline.py --episodes 50
```

## 4. Watch the trained agents

Loads the saved weights and runs with exploration off, which is how the policy would actually be deployed. One actor drives both rackets — the policy was trained by self-play and is symmetric.

In [ ]:
import torch

from tennis.agent import Agent
from tennis.config import Config

CHECKPOINT = "checkpoints/maddpg_seed0_solved.pt"

ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
agent = Agent(env.state_size, env.action_size, env.num_agents, Config.from_dict(ckpt["config"]))
agent.load(CHECKPOINT)

states = env.reset(train_mode=True)
totals = np.zeros(env.num_agents)
steps = 0

while steps < 1000:
    actions = agent.act(states, add_noise=False)
    states, rewards, dones, _ = env.step(actions)
    totals += rewards
    steps += 1
    if dones.any():
        break

print(f"episode score : {totals.max():.2f}   (target 0.5)")
print(f"rally length  : {steps} steps   (random policy: ~17)")

## 5. The learning curve

In [ ]:
import json

record = json.loads(open("results/maddpg_seed0.json").read())
print(f"solved at episode {record['solved_episode']} "
      f"({record['episodes_run']} episodes actually run)")
print(f"best 100-episode average: {record['best_moving_average']:.3f}")

In [ ]:
from IPython.display import Image

Image("assets/learning_curve.png")

Rally length is the mechanism behind that curve — the reward pays for keeping the ball in play, so the score and the rally are the same fact seen twice. It is also why the run gets slower as it improves: 0.14 s per episode early, 24.4 s after solving.

In [ ]:
Image("assets/rally_length.png")

## 6. Close the environment

In [ ]:
env.close()

---

## Training from the command line

```bash
python -m tennis.cli train --config configs/maddpg.yaml --seed 0
```

See [Report.md](Report.md) for the algorithm, hyperparameters, results and ideas for future work.